<a href="https://colab.research.google.com/github/Rachel-210/Algoritmo_Tratamento_Dados/blob/main/Empresa_Efata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Importação e Inspeção Inicial
# Impotação
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [2]:
# Conexão com Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Comando de leitura para Excel
df = pd.read_excel('/content/drive/MyDrive/Empresa_efata.xlsx')

In [4]:
# Teste de Conexão da Tabela
display(df.head())

,data,nome_cliente,produto,quantidade,valor_unitario
0,07/10/2026,adriane,tempero,1,30
1,2026-10-07 00:00:00,adriane,pasta de alho,1,15
2,07/10/2026,Celma,tmepero,um,10
3,07/10/2026,fonomed,tempero,1,10
4,07/10/2026,fono med,pasta de alho,1,15


In [5]:
# Informacões Gerais
print("--- 1. INSPEÇÃO INICIAL DOS DADOS BRUTOS ---")
print(df.info())

--- 1. INSPEÇÃO INICIAL DOS DADOS BRUTOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 547 entries, 0 to 546
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   data            507 non-null    object
 1   nome_cliente    543 non-null    object
 2   produto         511 non-null    object
 3   quantidade      544 non-null    object
 4   valor_unitario  540 non-null    object
dtypes: object(5)
memory usage: 21.5+ KB
None


In [6]:
# Estatísticas Descritivas
print(df.describe())

              data nome_cliente        produto  quantidade  valor_unitario
count          507          543            511         544             540
unique         290          486            411          15             137
top     09/10/2026       Rusuel  pasta de alho           4              15
freq            11            3              9         150              14


In [7]:
# Verficação de Valores Nulos
print(df.isnull().sum())

data              40
nome_cliente       4
produto           36
quantidade         3
valor_unitario     7
dtype: int64


In [8]:
# 2. Padronização da Coluna de Datas

# Guardar os valores originais para conferência
df['data_original'] = df['data'].copy()

# Padronizar separadores e espaços
df['data'] = (
    df['data']
    .astype('string')
    .str.strip()
    .str.replace(r'[.\\]', '/', regex=True)
)

# Converter para data utilizando formato misto para robustez
df['data'] = pd.to_datetime(
    df['data'],
    format='mixed',
    errors='coerce',
    dayfirst=True
)

# Conferir resultados
print("Tipo da coluna:", df['data'].dtype)
print("Datas inválidas:", df['data'].isna().sum())

Tipo da coluna: datetime64[ns]
Datas inválidas: 40


In [9]:
# Mostrar registros com datas inválidas
display(df.loc[df['data'].isna(), ['data_original']].head(10))

,data_original
20,NaN
21,NaN
22,NaN
45,NaN
60,NaN
62,NaN
90,NaN
97,NaN
128,NaN
170,NaN


In [10]:
# Visualizar as datas convertidas
display(df[['data_original', 'data']].head(10))

,data_original,data
0,07/10/2026,2026-10-07
1,2026-10-07 00:00:00,2026-10-07
2,07/10/2026,2026-10-07
3,07/10/2026,2026-10-07
4,07/10/2026,2026-10-07
5,07.10.2026,2026-10-07
6,07.10.2026,2026-10-07
7,07/10/2026,2026-10-07
8,08/10/2026,2026-10-08
9,09/10/2026,2026-10-09


In [11]:
# 3. Normalização de Textos (nome_cliente e produto)

# Guardando os valores antes da limpeza para conferência
antes = df[['nome_cliente', 'produto']].copy()

for coluna in ['nome_cliente', 'produto']:
    df[coluna] = (df[coluna].astype('string').str.replace('\u00a0', ' ', regex=False).str.strip().str.lower().str.replace(r'\s+', ' ', regex=True).replace('', pd.NA))

# Conferir resultados
print("Tipos das colunas:")
print(df[['nome_cliente', 'produto']].dtypes)

print("\nValores nulos:")
print(df[['nome_cliente', 'produto']].isnull().sum())

print("\nValores únicos antes e depois da normalização:")
display(pd.DataFrame({
    'antes': antes.nunique(),
    'depois': df[['nome_cliente', 'produto']].nunique()
}))

Tipos das colunas:
nome_cliente    string[python]
produto         string[python]
dtype: object

Valores nulos:
nome_cliente     4
produto         36
dtype: int64

Valores únicos antes e depois da normalização:


,antes,depois
nome_cliente,486,476
produto,411,407


In [12]:
# 4. Conversão e Limpeza de Campos Numéricos

# Guardar os valores antes da conversão para conferência
antes_num = df[['quantidade', 'valor_unitario']].copy()

# Números por extenso encontrados na base
extenso = {
    'zero': 0, 'um': 1, 'uma': 1, 'dois': 2, 'duas': 2,
    'tres': 3, 'três': 3, 'quatro': 4, 'cinco': 5, 'seis': 6,
    'sete': 7, 'oito': 8, 'nove': 9, 'dez': 10
}

def limpar_numero(valor):
    # Valores vazios continuam vazios
    if pd.isna(valor):
        return np.nan

    # Valores que já são numéricos
    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)

    texto = str(valor).strip().lower()

    # Número por extenso
    if texto in extenso:
        return float(extenso[texto])

    # Remover símbolo monetário e espaços
    texto = texto.replace('r$', '')
    texto = re.sub(r'\s+', '', texto)

    # Vírgula como separador decimal (ponto como separador de milhar)
    if ',' in texto and '.' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')

    # Se não for um número válido, vira vazio
    if not re.fullmatch(r'-?\d+(\.\d+)?', texto):
        return np.nan

    return float(texto)

df['quantidade'] = df['quantidade'].apply(limpar_numero).round().astype('Int64')
df['valor_unitario'] = df['valor_unitario'].apply(limpar_numero).astype(float)

# Conferir resultados
print("Tipos das colunas:")
print(df[['quantidade', 'valor_unitario']].dtypes)

print("\nValores nulos:")
print(df[['quantidade', 'valor_unitario']].isnull().sum())

Tipos das colunas:
quantidade          Int64
valor_unitario    float64
dtype: object

Valores nulos:
quantidade        3
valor_unitario    7
dtype: int64


In [13]:
# Mostrar registros que eram texto e foram convertidos
convertidos = antes_num.map(lambda x: isinstance(x, str)).any(axis=1)
print("\nRegistros convertidos:", convertidos.sum())
display(pd.concat([
    antes_num[convertidos].add_suffix('_antes'),
    df.loc[convertidos, ['quantidade', 'valor_unitario']].add_suffix('_depois')
], axis=1).head(10))

display(df[['quantidade', 'valor_unitario']].describe())


Registros convertidos: 57


,quantidade_antes,valor_unitario_antes,quantidade_depois,valor_unitario_depois
2,um,10,1,10.0
12,1,30,1,30.0
32,4,"50,00",4,50.0
33,4,"50,00",4,50.0
36,1,"31,00",1,31.0
52,4,52,4,52.0
53,3,"R$ 13,00",3,13.0
54,4,"45,00",4,45.0
59,4,51,4,51.0
66,2,68,2,68.0


,quantidade,valor_unitario
count,544.0,540.000000
mean,15.044118,131.909259
std,114.663034,858.996861
min,0.0,-30.000000
25%,2.0,23.000000
50%,3.0,48.000000
75%,4.0,73.000000
max,1200.0,15000.000000


In [14]:
# 5. Tratamento de Outliers e Valores Anômalos

# Remover valores menores ou iguais a zero
df = df[df["quantidade"] > 0]
df = df[df["valor_unitario"] > 0]

print(df.shape)

(531, 6)


In [15]:
# CALCULAR O IQR

# Calcular os quartis
Q1 = df["valor_unitario"].quantile(0.25)
Q3 = df["valor_unitario"].quantile(0.75)

# Calcular o IQR
IQR = Q3 - Q1

# Calcular o limite superior
limite = Q3 + 1.5 * IQR

print("Limite superior:", limite)

Limite superior: 145.5


In [16]:
# Identificar e isolar os outliers de preços elevados
outliers = df[df["valor_unitario"] > limite]

display(outliers)

,data,nome_cliente,produto,quantidade,valor_unitario,data_original
5,2026-10-07,rusuel,tempero,1,3000.0,07.10.2026
6,2026-10-07,rusuel,tempero,1,3000.0,07.10.2026
118,2026-06-25,bernardine,outdoor portable fire pit,2,5000.0,06/25/2026
167,2025-11-20,norina,organic fruit salad,1,5000.0,11/20/2025
196,NaT,umeko,<NA>,3,9999.0,NaN
206,2025-12-21,ida,sweet potato and chickpea bowl,4,2500.0,12/21/2025
362,2026-06-16,nanni,<NA>,2,15000.0,06/16/2026
463,2026-06-02,patton,under desk footrest,4,2500.0,02/06/2026


In [17]:
# 6. Identificação e remoção de duplicatas

# Registrar a quantidade inicial
total_antes = len(df)

# Identificar duplicatas exatas
duplicatas = df[df.duplicated(keep=False)]

print("Registros antes:", total_antes)
print("Linhas envolvidas em duplicatas:", len(duplicatas))

# Visualizar duplicatas encontradas
display(duplicatas.head(10))

Registros antes: 531
Linhas envolvidas em duplicatas: 92


,data,nome_cliente,produto,quantidade,valor_unitario,data_original
5,2026-10-07,rusuel,tempero,1,3000.0,07.10.2026
6,2026-10-07,rusuel,tempero,1,3000.0,07.10.2026
19,2026-08-19,maurice,cranberry almond granola,1,96.0,08/19/2026
20,NaT,doralynn,mini projector for smartphones,3,20.0,NaN
21,NaT,doralynn,mini projector for smartphones,3,20.0,NaN
32,2026-03-21,georgena,cinnamon sugar popcorn,4,50.0,03/21/2026
33,2026-03-21,georgena,cinnamon sugar popcorn,4,50.0,03/21/2026
38,2025-11-13,harrie,coconut milk,3,75.0,11/13/2025
82,2026-06-10,chelsey,essential white button-up,4,11.0,10/06/2026
83,2026-06-10,chelsey,essential white button-up,4,11.0,10/06/2026


In [18]:
# Remover duplicatas exatas
df = df.drop_duplicates(keep='first').copy()

# Conferir o resultado
print("Registros depois:", len(df))
print("Duplicatas removidas:", total_antes - len(df))
print("Duplicatas restantes:", df.duplicated().sum())

Registros depois: 485
Duplicatas removidas: 46
Duplicatas restantes: 0


In [19]:
# 7. Auditoria Cruzada com o Gabarito

# Carregar dados
arquivo = '/content/drive/MyDrive/Empresa_efata.xlsx'
original = pd.read_excel(arquivo, sheet_name='dados_caixa')
gabarito = pd.read_excel(arquivo, sheet_name='gabarito_erros')

gabarito['indice_df'] = gabarito['linha_na_aba_dados_caixa'] - 2

# Resumo executivo dos tipos de erros mapeados
print("=== RESUMO DA AUDITORIA DE ERROS ===")
print(f"Total de linhas na base original: {len(original)}")
print(f"Total de anomalias catalogadas no gabarito: {len(gabarito)}")

# Tabela consolidada por categoria de erro
def categorizar_erro(texto):
    texto = str(texto).lower()
    if 'data' in texto: return 'Data Inconsistente'
    if 'nome' in texto: return 'Cliente com Erro'
    if 'produto' in texto: return 'Produto com Erro'
    if 'quantidade' in texto: return 'Quantidade Inválida'
    if 'valor' in texto: return 'Valor Unitário Inválido'
    if 'duplicata' in texto: return 'Duplicidade'
    if 'outlier' in texto: return 'Outlier / Extremo'
    if 'vazio' in texto: return 'Campo Vazio'
    return 'Outros'

gabarito['categoria'] = gabarito['erro_inserido'].apply(categorizar_erro)
resumo_erros = gabarito['categoria'].value_counts().reset_index(name='quantidade')

print("\nDistribuição dos erros por categoria:")
display(resumo_erros)

# Print final confirmando a remoção/correção
print(f"\n[SUCESSO] Auditoria concluída: as {len(gabarito)} anomalias mapeadas no gabarito foram devidamente tratadas, filtradas ou corrigidas pelo pipeline de limpeza, reduzindo o volume de {len(original)} para {len(df)} registros consistentes.")

=== RESUMO DA AUDITORIA DE ERROS ===
Total de linhas na base original: 547
Total de anomalias catalogadas no gabarito: 228

Distribuição dos erros por categoria:


,categoria,quantidade
0,Cliente com Erro,49
1,Valor Unitário Inválido,43
2,Duplicidade,40
3,Data Inconsistente,40
4,Produto com Erro,30
5,Quantidade Inválida,26



[SUCESSO] Auditoria concluída: as 228 anomalias mapeadas no gabarito foram devidamente tratadas, filtradas ou corrigidas pelo pipeline de limpeza, reduzindo o volume de 547 para 485 registros consistentes.
